In [0]:
%run ../control_framework/__init__ 

In [0]:
import time
from pyspark.sql import Row

In [0]:
dbutils.widgets.text("env", "dev", "env")
# dbutils.widgets.text("param_name", "ingestion", "param name")
dbutils.widgets.text("process_name", "finnhub_dp_monthly_load", "process name")
dbutils.widgets.text("process_date", "20260716", "process date")
dbutils.widgets.text("run_type", "load", "run type")
dbutils.widgets.text("source", "m101_finnhub", "Source")
dbutils.widgets.text("target", "finnhub_dp", "Target")
dbutils.widgets.text("execution_type", "consumption", "execution type")

In [0]:
env = dbutils.widgets.get("env")
# param_name = dbutils.widgets.get("param_name")
process_name = dbutils.widgets.get("process_name")
process_date = dbutils.widgets.get("process_date")
process_date = int(process_date)
run_type = dbutils.widgets.get("run_type")
source = dbutils.widgets.get("source")
target = dbutils.widgets.get("target")
execution_type = dbutils.widgets.get("execution_type")
print(f"env: {env}")
# print(f"param_name: {param_name}")
print(f"process_name: {process_name}")
print(f"process_date: {process_date}")
print(f"run_type: {run_type}")
print(f"source: {source}")
print(f"target: {target}")
print(f"execution_type: {execution_type}")

In [0]:
try:
    comment = register_tgt(target, execution_type)
    insert_log(target, process_name, "target_register","target_register","target_register","target_register",process_date, 'success',comment)
except Exception as e:
    error_msg = str(e).replace("'", "''")
    insert_log(target, process_name, "target_register","target_register","target_register","target_register",process_date, 'failed', error_msg)
    print(f"Error in register_target: {e}")
    dbutils.notebook.exit(f"Error in register_target: {e}")

In [0]:
src_schema1 = f_get_src_schema(source)
src_tbl1 = 'finnhubb_monthly_stock_financial_ts'
# src_schema_master = f_get_src_schema('master')
# src_master_tb1 = 's_and_p_500_dev'
target_schema = f_get_tgt_schema(target)
target_tbl = 'finnhubb_monthly_stock_financial'
full_target_table_name = f"{target_schema}.{target_tbl}"
print(src_schema1)
print(src_tbl1)
# print(src_schema_master)
# print(src_master_tb1)
print(target_schema)
print(target_tbl)
print(full_target_table_name)

In [0]:
src_tbl_list = [{"source":source,"source_schema":src_schema1,"source_table": src_tbl1}]
tgt_tbl_list = [{"target":target,"target_schema":target_schema,"target_table": target_tbl,'holiday_ind': 'N','weekend_ind': 'N','calender':'','frequency': '01_monthly'}]

In [0]:
src_tgt_mapping_list = [
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'ticker','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'stock_ticker','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'name','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'org_name','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'country','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'org_base_country','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'currency','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'managing_currency','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'estimateCurrency','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'traded_currency','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'exchange','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'exchange_name','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'ipo','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'ipo_date','target_column_datatype':'date'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'marketCapitalization','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'market_capitalization','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'logo','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'org_logo_url','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'shareOutstanding','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'share_outstanding','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'finnhubIndustry','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'industry_type','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'phone','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'org_phone_no','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'weburl','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'website_address','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'floatingShare','source_column_datatype':'double','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'floating_share','target_column_datatype':'double'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'symbol','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'stock_symbol','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'process_date','source_column_datatype':'long','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'process_date','target_column_datatype':'long'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'r_source','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'r_source','target_column_datatype':'string'},
    {'source_schema':src_schema1,'source_table':src_tbl1,'source_column_name':'hash_id','source_column_datatype':'string','target_schema':target_schema,'target_table':target_tbl,'target_column_name':'hash_id','target_column_datatype':'string'}
]

In [0]:
if run_type == 'load_metadata':
    status = run_load_metadata(src_tbl_list,tgt_tbl_list,target,process_name,src_tgt_mapping_list,process_date)
    dbutils.notebook.exit(status)


In [0]:
from pyspark.sql.types import StringType, DoubleType, LongType, StructType, StructField, DateType

schema_structure = {
    "stock_ticker": StringType(),
    "org_name": StringType(),
    "org_base_country": StringType(),
    "managing_currency": StringType(),
    "traded_currency": StringType(),
    "exchange_name": StringType(),
    "ipo_date": DateType(),
    "market_capitalization": DoubleType(),
    "org_logo_url": StringType(),
    "share_outstanding": DoubleType(),
    "industry_type": StringType(),
    "org_phone_no": StringType(),
    "website_address": StringType(),
    "floating_share": DoubleType(),
    "stock_symbol": StringType(),
    "process_date": LongType(),
    "r_source": StringType(),
    "hash_id": StringType(),
    "r_target": StringType()
}

In [0]:
delta_logic = f"""process_date = {process_date}"""

In [0]:
if run_type == 'full':
    create_tgt_tbl_log = create_target_table(full_target_table_name,schema_structure)
    if  create_tgt_tbl_log == f"Table {target_schema}.{target_tbl} created successfully":
        process_status = 'success'
    else:
        process_status = 'failed'
    log = str(create_tgt_tbl_log).replace("'", "''")
    insert_log (target, process_name, 'create_tgt_tbl','create_tgt_tbl', target_schema, target_tbl, process_date,process_status, log)

    delta_logic = f"""process_date <= {process_date}"""

In [0]:
column_fetch = """
ticker as stock_ticker,
name as org_name,
country as org_base_country,
currency as managing_currency,
estimateCurrency as traded_currency,
exchange as exchange_name,
CAST(ipo AS DATE) as ipo_date,
marketCapitalization as market_capitalization,
logo as org_logo_url,
shareOutstanding as share_outstanding,
finnhubIndustry as industry_type,
phone as org_phone_no,
weburl as website_address,
floatingShare as floating_share,
symbol as stock_symbol,
process_date,
r_source,
hash_id
"""

In [0]:
v_sel_main1 = f"""with s as (
  select *, row_number() over (partition by hash_id order by process_date desc) as rn, '{target}' as r_target
  from {src_schema1}.{src_tbl1}
  where {delta_logic}
),
src as (
  select {column_fetch} from s where rn = 1
),
tgt as (
  select * from {target_schema}.{target_tbl}
  where r_target = '{target}'
),
main as (
  select src.*
  from src
  left anti join tgt
  on src.hash_id = tgt.hash_id
)
select *,'{target}' as r_target  from main
"""
print(v_sel_main1)

In [0]:
insert_table_query = f"""insert into {target_schema}.{target_tbl}
(stock_ticker,
  org_name,
  org_base_country,
  managing_currency,
  traded_currency,
  exchange_name,
  ipo_date,
  market_capitalization,
  org_logo_url,
  share_outstanding,
  industry_type,
  org_phone_no,
  website_address,
  floating_share,
  stock_symbol,
  process_date,
  r_source,
  hash_id,
  r_target)
 {v_sel_main1}"""
print(insert_table_query)

In [0]:
if run_type == "rollback":
    try:
        comment = rollback_run(tgt_tbl_list,process_date)
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'success', str(comment).replace("'", "''"))
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'rollback', 'rollback', target_schema, target_tbl, process_date, 'failed', error_msg)
        comment = f"Error in rollback: {e}"
    dbutils.notebook.exit(f"{comment}")

In [0]:
if run_type == 'full' or run_type == 'load':
    try:
        # data_list_val = data_list(symbol_list)
        # insert_ingestion_table_query(data_list_val, target_schema, target_tbl)
        # data_df = get_data(data_list_val,schema_structure)
        # anomoly_df = validate_and_store_anomalies(data_df,schema_structure,full_target_table_name)
        # clean_df = remove_anomalies_by_hash_id(data_df, anomoly_df)
        spark.sql(f"""{insert_table_query}""")
        insert_log(target, process_name, 'data_consumption', 'data_consumption', target_schema, target_tbl, process_date, 'success', 'Data consumption completed')
        e = "success"
    except Exception as e:
        error_msg = str(e).replace("'", "''")
        insert_log(target, process_name, 'data_consumption', 'data_consumption', target_schema, target_tbl, process_date, 'failed', error_msg)
        e = f"Error in data_consumption: {e}"
    dbutils.notebook.exit(f"{e}")